# Analisis Spasial dan Temporal Ketimpangan Kualitas Udara
## di 10 Kota Besar Indonesia Berbasis Data Analytics

Notebook ini menjalankan **Data Cleaning, Exploratory Data Analysis (EDA), visualisasi, dan ekstraksi insight** dari dataset kualitas udara.

**Cara menggunakan:**
1. Jalankan setiap cell secara berurutan.
2. Saat diminta, unggah file `indonesia_air_quality_health (2).csv` (atau file CSV dataset yang digunakan).
3. Setelah analisis selesai, unduh ZIP berisi seluruh hasil analisis dari cell terakhir.

> Catatan: hasil analisis bergantung pada isi dataset yang diunggah. Program ini tidak menggunakan Machine Learning.

## 1. Instalasi dan Import Library

In [ ]:
!pip -q install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from google.colab import files
import zipfile
from IPython.display import display

sns.set_theme(style="whitegrid")
folder_output = Path("hasil_analisis")
folder_output.mkdir(exist_ok=True)

print("Library berhasil disiapkan.")

## 2. Upload dan Membaca Dataset

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Belum ada file yang diunggah.")

nama_file = next(iter(uploaded))
df = pd.read_csv(nama_file)

# Memperbaiki nama kolom ID jika diperlukan
if "h" in df.columns and "record_id" not in df.columns:
    df.rename(columns={"h": "record_id"}, inplace=True)

print("Nama file:", nama_file)
print("Jumlah data (baris, kolom):", df.shape)
display(df.head())

kolom_wajib = [
    "timestamp", "city", "zone_type", "temperature_c",
    "rainfall_mm", "traffic_index", "pm25_ug_m3",
    "no2_ppb", "aqi_score", "respiratory_cases"
]
kolom_tidak_ada = [k for k in kolom_wajib if k not in df.columns]
if kolom_tidak_ada:
    raise ValueError(f"Kolom berikut tidak ditemukan dalam dataset: {kolom_tidak_ada}")

## 3. Data Cleaning

In [ ]:
print("Informasi Dataset:")
df.info()

print("\nMissing Values Sebelum Cleaning:")
display(df.isnull().sum().to_frame("jumlah_missing"))

if "record_id" in df.columns:
    df = df.drop_duplicates(subset="record_id")
df = df.drop_duplicates()

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

kolom_numerik = [
    "temperature_c", "rainfall_mm", "traffic_index",
    "pm25_ug_m3", "no2_ppb", "aqi_score", "respiratory_cases"
]

for kolom in kolom_numerik:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce")

# Mengisi missing values berdasarkan median kota dan zona,
# kemudian menggunakan median seluruh kolom sebagai cadangan.
for kolom in kolom_numerik:
    median_kelompok = df.groupby(["city", "zone_type"])[kolom].transform("median")
    df[kolom] = df[kolom].fillna(median_kelompok)
    if df[kolom].notna().any():
        df[kolom] = df[kolom].fillna(df[kolom].median())

df["zone_type"] = df["zone_type"].fillna("Tidak diketahui")
df["air_quality_status"] = (
    df["air_quality_status"].fillna("Tidak diketahui")
    if "air_quality_status" in df.columns
    else "Tidak diketahui"
)

df = df.dropna(subset=["timestamp", "city"]).copy()

df["tanggal"] = df["timestamp"].dt.date
df["jam"] = df["timestamp"].dt.hour
df["hari"] = df["timestamp"].dt.day_name()
df["aqi_risk"] = np.where(df["aqi_score"] > 100, "Berisiko", "Tidak Berisiko")

df.to_csv(folder_output / "dataset_bersih.csv", index=False)

print("Data Cleaning Selesai")
print("Jumlah data bersih:", len(df))
print("\nMissing Values Setelah Cleaning:")
display(df.isnull().sum().to_frame("jumlah_missing"))

## 4. EDA Spasial Berdasarkan Kota

In [ ]:
analisis_kota = df.groupby("city").agg(
    jumlah_data=("aqi_score", "count"),
    rata_rata_aqi=("aqi_score", "mean"),
    rata_rata_pm25=("pm25_ug_m3", "mean"),
    rata_rata_no2=("no2_ppb", "mean"),
    rata_rata_ispa=("respiratory_cases", "mean"),
    persentase_risiko=("aqi_risk", lambda x: (x == "Berisiko").mean() * 100)
).round(2).sort_values("rata_rata_aqi", ascending=False)

print("Analisis Kualitas Udara Per Kota:")
display(analisis_kota)

analisis_kota.to_csv(folder_output / "analisis_per_kota.csv")

## 5. EDA Berdasarkan Zona

In [ ]:
analisis_zona = df.groupby("zone_type").agg(
    jumlah_data=("aqi_score", "count"),
    rata_rata_aqi=("aqi_score", "mean"),
    rata_rata_pm25=("pm25_ug_m3", "mean"),
    rata_rata_no2=("no2_ppb", "mean"),
    rata_rata_traffic=("traffic_index", "mean"),
    rata_rata_ispa=("respiratory_cases", "mean")
).round(2)

print("Analisis Berdasarkan Zona:")
display(analisis_zona)

analisis_zona.to_csv(folder_output / "analisis_per_zona.csv")

## 6. EDA Temporal Berdasarkan Jam

In [ ]:
analisis_jam = df.groupby("jam").agg(
    rata_rata_aqi=("aqi_score", "mean"),
    rata_rata_pm25=("pm25_ug_m3", "mean"),
    rata_rata_no2=("no2_ppb", "mean"),
    rata_rata_ispa=("respiratory_cases", "mean")
).round(2)

print("Analisis Berdasarkan Jam:")
display(analisis_jam)

analisis_jam.to_csv(folder_output / "analisis_per_jam.csv")

## 7. EDA Temporal Berdasarkan Tanggal

In [ ]:
analisis_harian = df.groupby("tanggal").agg(
    rata_rata_aqi=("aqi_score", "mean"),
    rata_rata_pm25=("pm25_ug_m3", "mean"),
    jumlah_pengamatan=("aqi_score", "count")
).round(2)

print("Analisis Harian:")
display(analisis_harian.head(10))

analisis_harian.to_csv(folder_output / "analisis_harian.csv")

## 8. Visualisasi Data

In [ ]:
# Grafik perbandingan AQI antar kota
plt.figure(figsize=(12, 6))
sns.barplot(data=analisis_kota.reset_index(), x="city", y="rata_rata_aqi")
plt.axhline(100, color="red", linestyle="--", label="Ambang AQI 100")
plt.title("Perbandingan Rata-rata AQI di Kota dalam Dataset")
plt.xlabel("Kota")
plt.ylabel("Rata-rata AQI")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig(folder_output / "grafik_aqi_kota.png", dpi=150)
plt.show()

# Grafik AQI berdasarkan zona
plt.figure(figsize=(9, 5))
sns.barplot(data=analisis_zona.reset_index(), x="zone_type", y="rata_rata_aqi")
plt.axhline(100, color="red", linestyle="--", label="Ambang AQI 100")
plt.title("Rata-rata AQI Berdasarkan Zona")
plt.xlabel("Jenis Zona")
plt.ylabel("Rata-rata AQI")
plt.xticks(rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig(folder_output / "grafik_aqi_zona.png", dpi=150)
plt.show()

# Pola AQI berdasarkan jam
plt.figure(figsize=(10, 5))
sns.lineplot(data=analisis_jam.reset_index(), x="jam", y="rata_rata_aqi", marker="o")
plt.axhline(100, color="red", linestyle="--", label="Ambang AQI 100")
plt.title("Pola AQI Berdasarkan Jam Pengamatan")
plt.xlabel("Jam")
plt.ylabel("Rata-rata AQI")
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.savefig(folder_output / "grafik_aqi_jam.png", dpi=150)
plt.show()

# Tren AQI harian
harian_plot = analisis_harian.reset_index()
harian_plot["tanggal"] = pd.to_datetime(harian_plot["tanggal"])
plt.figure(figsize=(12, 5))
sns.lineplot(data=harian_plot, x="tanggal", y="rata_rata_aqi", marker="o")
plt.axhline(100, color="red", linestyle="--", label="Ambang AQI 100")
plt.title("Tren Kualitas Udara Harian")
plt.xlabel("Tanggal")
plt.ylabel("Rata-rata AQI")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(folder_output / "grafik_tren_harian.png", dpi=150)
plt.show()

## 9. Heatmap AQI Berdasarkan Kota dan Jam

In [ ]:
matriks_aqi = df.pivot_table(
    index="city", columns="jam", values="aqi_score", aggfunc="mean"
)

plt.figure(figsize=(13, 7))
sns.heatmap(matriks_aqi, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Heatmap AQI Berdasarkan Kota dan Jam")
plt.xlabel("Jam Pengamatan")
plt.ylabel("Kota")
plt.tight_layout()
plt.savefig(folder_output / "heatmap_aqi.png", dpi=150)
plt.show()

## 10. Korelasi Antarvariabel

In [ ]:
kolom_korelasi = [
    "temperature_c", "rainfall_mm", "traffic_index",
    "pm25_ug_m3", "no2_ppb", "aqi_score", "respiratory_cases"
]

korelasi = df[kolom_korelasi].corr(method="spearman")

plt.figure(figsize=(10, 7))
sns.heatmap(korelasi, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Korelasi Antarvariabel Kualitas Udara (Spearman)")
plt.tight_layout()
plt.savefig(folder_output / "heatmap_korelasi.png", dpi=150)
plt.show()

display(korelasi)

## 11. Ekstraksi Insight

In [ ]:
if analisis_kota.empty or analisis_zona.empty or analisis_jam.empty:
    raise ValueError("Data tidak cukup untuk menghitung insight.")

kota_tertinggi = analisis_kota["rata_rata_aqi"].idxmax()
kota_terendah = analisis_kota["rata_rata_aqi"].idxmin()
zona_tertinggi = analisis_zona["rata_rata_aqi"].idxmax()
jam_tertinggi = analisis_jam["rata_rata_aqi"].idxmax()
persentase_risiko = (df["aqi_score"] > 100).mean() * 100

baris_insight = [
    "HASIL EKSTRAKSI INSIGHT",
    "=======================",
    f"Kota AQI tertinggi: {kota_tertinggi}",
    f"Kota AQI terendah: {kota_terendah}",
    f"Zona AQI tertinggi: {zona_tertinggi}",
    f"Jam AQI tertinggi: {jam_tertinggi}:00",
    f"Persentase AQI > 100: {persentase_risiko:.2f}%"
]

print("\n".join(baris_insight))

with open(folder_output / "hasil_insight.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(baris_insight) + "\n")

## 12. Unduh Seluruh Hasil Analisis

In [ ]:
# Membuat ZIP dari seluruh file hasil analisis
zip_path = Path("hasil_analisis.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in folder_output.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, arcname=file_path.relative_to(folder_output.parent))

print("Analisis selesai. File hasil tersedia dalam:", zip_path)
files.download(str(zip_path))

## Selesai

Tahapan yang telah dilakukan:
1. Data Cleaning
2. Exploratory Data Analysis (EDA)
3. Visualisasi Data
4. Ekstraksi Insight

**Catatan interpretasi:** garis AQI 100 digunakan mengikuti batas operasional yang ditetapkan dalam kode ini. Pastikan definisi kategori AQI sesuai dengan standar dan sumber data yang digunakan sebelum menyusun kesimpulan akhir.